In [1]:
import sys, os, time, getpass

sys.path.append("/home/tatiane/lib/")

import pessoal
from pessoal import *

spark.sparkContext.setLogLevel("ERROR")

Tempo inicial da execucao: 2025-11-23 19:01:16.182421
User: tatiane
Node: tatiane-Inspiron-3583


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/23 19:01:21 WARN Utils: Your hostname, tatiane-Inspiron-3583, resolves to a loopback address: 127.0.1.1; using 192.168.0.14 instead (on interface wlo1)
25/11/23 19:01:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/23 19:01:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/23 19:01:25 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/11/23 19:01:25 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


## Leitura das bases

In [2]:
pnadc_2023 = spark.read.parquet("/home/tatiane/Downloads/NINSOC/PNADC/ano=2023/")
pnadc_2024 = spark.read.parquet("/home/tatiane/Downloads/NINSOC/PNADC/ano=2024/")
pnadc_2025_3 = spark.read.parquet("/home/tatiane/Downloads/NINSOC/PNADC/ano=2025/")

In [3]:
print("Completude do ano de 2023: ")
pessoal.completudeSchema(pnadc_2023)

print("Completude do ano de 2024: ")
pessoal.completudeSchema(pnadc_2024)

print("Completude do ano de 2025 - até o 3º trimestre: ")
pessoal.completudeSchema(pnadc_2025_3)

Completude do ano de 2023: 


Qtd. registros: 1900989 | Quantidade de colunas:  14
root
 |-- ano: integer (nullable = true)
 |-- sexo: string (nullable = true)
 |-- raca_cor: string (nullable = true)
 |-- idade_dt_referencia: integer (nullable = true)
 |-- qtd_pessoa_domicilio: integer (nullable = true)
 |-- condicao_domicilio: string (nullable = true)
 |-- situacao_domicilio: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- rmh_todos_trabalhos: float (nullable = true)
 |-- peso_domicilio_pessoa: float (nullable = true)
 |-- tp_remuneracao_habitual: string (nullable = true)
 |-- id_unico: long (nullable = true)
 |-- rendimento_habitual: string (nullable = true)
 |-- recebe_remuneracao: string (nullable = true)

Completude do ano de 2024: 
Qtd. registros: 1910447 | Quantidade de colunas:  14
root
 |-- ano: integer (nullable = true)
 |-- sexo: string (nullable = true)
 |-- raca_cor: string (nullable = true)
 |-- idade_dt_referencia: integer (nullable = true)
 |-- qtd_pessoa_domicilio: integer (nullable

### Bind

In [3]:
pnadc_2023_2025_3 = pnadc_2023.unionByName(pnadc_2024).unionByName(pnadc_2025_3)
pessoal.completudeSchema(pnadc_2023_2025_3)

[Stage 3:===================================================>     (10 + 1) / 11]

Qtd. registros: 5249881 | Quantidade de colunas:  14
root
 |-- ano: integer (nullable = true)
 |-- sexo: string (nullable = true)
 |-- raca_cor: string (nullable = true)
 |-- idade_dt_referencia: integer (nullable = true)
 |-- qtd_pessoa_domicilio: integer (nullable = true)
 |-- condicao_domicilio: string (nullable = true)
 |-- situacao_domicilio: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- rmh_todos_trabalhos: float (nullable = true)
 |-- peso_domicilio_pessoa: float (nullable = true)
 |-- tp_remuneracao_habitual: string (nullable = true)
 |-- id_unico: long (nullable = true)
 |-- rendimento_habitual: string (nullable = true)
 |-- recebe_remuneracao: string (nullable = true)



#### Criação de variáveis

In [4]:
pnadc_2023_2025_3 = pnadc_2023_2025_3.withColumn(
    "rmh_todos_trabalhos",
    F.when(F.col("rmh_todos_trabalhos").isNull(), F.lit(0))
     .otherwise(F.col("rmh_todos_trabalhos"))
)

In [6]:
pnadc_2023_2025_3 = pnadc_2023_2025_3.withColumn("renda_pc", F.col("rmh_todos_trabalhos") / F.col("qtd_pessoa_domicilio"))

In [8]:
pnadc_2023_2025_3 = pnadc_2023_2025_3.withColumn('ind_pobreza', F.when(F.col("renda_pc") < 665, 1).otherwise(0))

In [10]:
pessoal.completudeVar(pnadc_2023_2025_3)

Resumo do DataFrame:


,coluna,total,duplicados,nulos,valores_unicos
0,sexo,5249881,5249879,0,2
1,situacao_domicilio,5249881,5249879,0,2
2,ind_pobreza,5249881,5249879,0,2
3,recebe_remuneracao,5249881,5249879,0,2
4,rendimento_habitual,5249881,5249879,0,2
5,tp_remuneracao_habitual,5249881,5249878,2952524,3
6,ano,5249881,5249878,0,3
7,raca_cor,5249881,5249875,0,6
8,condicao_domicilio,5249881,5249862,0,19
9,qtd_pessoa_domicilio,5249881,5249860,0,21


### Analisando a distribuição das variáveis

In [11]:
for col in pnadc_2023_2025_3:
    pnadc_2023_2025_3.groupBy(col).count().show()

+----+-------+
| ano|  count|
+----+-------+
|2023|1900989|
|2024|1910447|
|2025|1438445|
+----+-------+



+------+-------+
|  sexo|  count|
+------+-------+
| Homem|2534967|
|Mulher|2714914|
+------+-------+



+--------+-------+
|raca_cor|  count|
+--------+-------+
|Ignorado|    502|
|   Preta| 516115|
|Indigena|  30701|
| Amarela|  26138|
|  Branca|2058331|
|   Parda|2618094|
+--------+-------+

+-------------------+-----+
|idade_dt_referencia|count|
+-------------------+-----+
|                 31|67822|
|                 85|10933|
|                 65|55700|
|                 53|67527|
|                 78|22648|
|                108|   20|
|                 34|70856|
|                101|  214|
|                115|    3|
|                 81|16152|
|                 28|71440|
|                 76|26690|
|                 26|69842|
|                 27|70874|
|                 44|76575|
|                103|  132|
|                 12|73741|
|                 91| 3923|
|                 22|69772|
|                 93| 2762|
+-------------------+-----+
only showing top 20 rows
+--------------------+-------+
|qtd_pessoa_domicilio|  count|
+--------------------+-------+
|                  

+--------------------+-------+
|  condicao_domicilio|  count|
+--------------------+-------+
|         Pensionista|    171|
|Pessoa responsáve...|1940021|
|          Bisneto(a)|   7109|
|Cônjuge do mesmo ...|   8056|
|Convivente - Comp...|  13739|
|Filho(a) só do re...| 608629|
|Parente do(a) emp...|    113|
|Empregado(a) domé...|   1037|
|Agregado(a) - Não...|   9413|
|            Sogro(a)|  14382|
|       Genro ou nora|  39114|
|Cônjuge de sexo d...|1097020|
|   Filho(a) de ambos| 977490|
|             Neto(a)| 205447|
|          Avô ou avó|   6013|
|       Outro parente|  67017|
|             Irmã(o)|  82985|
|Pai, mãe, padrast...| 110905|
|          Enteado(a)|  61220|
+--------------------+-------+

+------------------+-------+
|situacao_domicilio|  count|
+------------------+-------+
|            Urbana|3845686|
|             Rural|1404195|
+------------------+-------+

+---+------+
| uf| count|
+---+------+
| 51|139247|
| 15|191104|
| 11| 85675|
| 29|250913|
| 42|331549|
| 28| 9

+-------------------+-----+
|rmh_todos_trabalhos|count|
+-------------------+-----+
|             4800.0| 3007|
|             3980.0|   30|
|           300000.0|    9|
|              305.0|   13|
|             1051.0|    6|
|             5360.0|    8|
|            20948.0|    1|
|              769.0|    6|
|             1761.0|    5|
|              558.0|    5|
|             6433.0|    3|
|              496.0|    3|
|             2862.0|    5|
|            20893.0|    1|
|             3597.0|    2|
|              720.0| 1415|
|             3029.0|    7|
|              810.0|   86|
|            55833.0|    1|
|            21500.0|   55|
+-------------------+-----+
only showing top 20 rows


...

+--------------------+-------+
| rendimento_habitual|  count|
+--------------------+-------+
|Com rendimento ha...|2241900|
|       Não aplicável|3007981|
+--------------------+-------+



+------------------+-------+
|recebe_remuneracao|  count|
+------------------+-------+
|        Não recebe|2952524|
|            Recebe|2297357|
+------------------+-------+



+------------------+-----+
|          renda_pc|count|
+------------------+-----+
|             596.0|   18|
|             934.0|   24|
|             305.0|  105|
|            3980.0|    9|
|            4800.0|  411|
|1400.6666666666667|    1|
|             692.0|   15|
|             558.0|   10|
|           10625.0|    2|
|             496.0|   17|
|             299.8|    1|
|            1051.0|   13|
|             451.5|    7|
| 467.6666666666667|    1|
|            1761.0|    2|
|              15.5|    1|
|             769.0|   13|
|             330.4|    5|
| 574.6666666666666|   10|
|            367.25|    2|
+------------------+-----+
only showing top 20 rows


[Stage 347:=============================================>          (9 + 2) / 11]

+-----------+-------+
|ind_pobreza|  count|
+-----------+-------+
|          1|4148463|
|          0|1101418|
+-----------+-------+



### Escrita da base

In [12]:
pnadc_2023_2025_3.coalesce(1).write.mode("overwrite").parquet("/home/tatiane/Downloads/NINSOC/base_final/pnadc_2023_2025_3")

### Finalização do notebook

In [13]:
executionTime()

Tempo de execucao ate este ponto: 0:14:34.326756
